In [ ]:
import os
import sys
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ----- Configuration -----
REPO_ROOT = "/home/iec/MinhHieu/rPPG"
if not os.path.exists(REPO_ROOT):
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
PREPROCESSED_PATH = os.path.join(REPO_ROOT, "preprocessed_data/Headmotion/groupC")
OUTPUT_WEIGHTS_DIR = os.path.join(REPO_ROOT, "final_model_release")
OUTPUT_WEIGHTS_PATH = os.path.join(OUTPUT_WEIGHTS_DIR, "GroupC_PhysNet.pth")

os.makedirs(OUTPUT_WEIGHTS_DIR, exist_ok=True)

CHUNK_LENGTH = 128
BATCH_SIZE = 4
EPOCHS = 10
LR = 1e-4
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"Weights will be saved to: {OUTPUT_WEIGHTS_PATH}")
import csv
import time
import math


## Inlined source

The cells below contain the model / loss source that was previously imported from the `neural_methods/` (and `evaluation/`) packages. They are inlined here so the notebook is self-contained.


In [ ]:
# === inlined from neural_methods/model/PhysNet.py ===
""" PhysNet
We repulicate the net pipeline of the orginal paper, but set the input as diffnormalized data.
orginal source:
Remote Photoplethysmograph Signal Measurement from Facial Videos Using Spatio-Temporal Networks
British Machine Vision Conference (BMVC)} 2019,
By Zitong Yu, 2019/05/05
Only for research purpose, and commercial use is not allowed.
MIT License
Copyright (c) 2019
"""

import math
import pdb

import torch
import torch.nn as nn
from torch.nn.modules.utils import _triple


class PhysNet_padding_Encoder_Decoder_MAX(nn.Module):
    def __init__(self, frames=128):
        super(PhysNet_padding_Encoder_Decoder_MAX, self).__init__()

        self.ConvBlock1 = nn.Sequential(
            nn.Conv3d(3, 16, [1, 5, 5], stride=1, padding=[0, 2, 2]),
            nn.BatchNorm3d(16),
            nn.ReLU(inplace=True),
        )

        self.ConvBlock2 = nn.Sequential(
            nn.Conv3d(16, 32, [3, 3, 3], stride=1, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
        )
        self.ConvBlock3 = nn.Sequential(
            nn.Conv3d(32, 64, [3, 3, 3], stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
        )

        self.ConvBlock4 = nn.Sequential(
            nn.Conv3d(64, 64, [3, 3, 3], stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
        )
        self.ConvBlock5 = nn.Sequential(
            nn.Conv3d(64, 64, [3, 3, 3], stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
        )
        self.ConvBlock6 = nn.Sequential(
            nn.Conv3d(64, 64, [3, 3, 3], stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
        )
        self.ConvBlock7 = nn.Sequential(
            nn.Conv3d(64, 64, [3, 3, 3], stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
        )
        self.ConvBlock8 = nn.Sequential(
            nn.Conv3d(64, 64, [3, 3, 3], stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
        )
        self.ConvBlock9 = nn.Sequential(
            nn.Conv3d(64, 64, [3, 3, 3], stride=1, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
        )

        self.upsample = nn.Sequential(
            nn.ConvTranspose3d(in_channels=64, out_channels=64, kernel_size=[
                4, 1, 1], stride=[2, 1, 1], padding=[1, 0, 0]),  # [1, 128, 32]
            nn.BatchNorm3d(64),
            nn.ELU(),
        )
        self.upsample2 = nn.Sequential(
            nn.ConvTranspose3d(in_channels=64, out_channels=64, kernel_size=[
                4, 1, 1], stride=[2, 1, 1], padding=[1, 0, 0]),  # [1, 128, 32]
            nn.BatchNorm3d(64),
            nn.ELU(),
        )

        self.ConvBlock10 = nn.Conv3d(64, 1, [1, 1, 1], stride=1, padding=0)

        self.MaxpoolSpa = nn.MaxPool3d((1, 2, 2), stride=(1, 2, 2))
        self.MaxpoolSpaTem = nn.MaxPool3d((2, 2, 2), stride=2)

        # self.poolspa = nn.AdaptiveMaxPool3d((frames,1,1))    # pool only spatial space
        self.poolspa = nn.AdaptiveAvgPool3d((frames, 1, 1))

    def forward(self, x):  # Batch_size*[3, T, 128,128]
        x_visual = x
        [batch, channel, length, width, height] = x.shape

        x = self.ConvBlock1(x)  # x [3, T, 128,128]
        x = self.MaxpoolSpa(x)  # x [16, T, 64,64]

        x = self.ConvBlock2(x)  # x [32, T, 64,64]
        x_visual6464 = self.ConvBlock3(x)  # x [32, T, 64,64]
        # x [32, T/2, 32,32]    Temporal halve
        x = self.MaxpoolSpaTem(x_visual6464)

        x = self.ConvBlock4(x)  # x [64, T/2, 32,32]
        x_visual3232 = self.ConvBlock5(x)  # x [64, T/2, 32,32]
        x = self.MaxpoolSpaTem(x_visual3232)  # x [64, T/4, 16,16]

        x = self.ConvBlock6(x)  # x [64, T/4, 16,16]
        x_visual1616 = self.ConvBlock7(x)  # x [64, T/4, 16,16]
        x = self.MaxpoolSpa(x_visual1616)  # x [64, T/4, 8,8]

        x = self.ConvBlock8(x)  # x [64, T/4, 8, 8]
        x = self.ConvBlock9(x)  # x [64, T/4, 8, 8]
        x = self.upsample(x)  # x [64, T/2, 8, 8]
        x = self.upsample2(x)  # x [64, T, 8, 8]

        # x [64, T, 1,1]    -->  groundtruth left and right - 7
        x = self.poolspa(x)
        x = self.ConvBlock10(x)  # x [1, T, 1,1]

        rPPG = x.view(-1, length)

        return rPPG, x_visual, x_visual3232, x_visual1616


In [ ]:
# === inlined from neural_methods/loss/PhysNetNegPearsonLoss.py ===
from __future__ import print_function, division
import torch
import matplotlib.pyplot as plt
import argparse, os
import pandas as pd
import numpy as np
import random
import math
from torchvision import transforms
from torch import nn


class Neg_Pearson(nn.Module):
    """
    The Neg_Pearson Module is from the orignal author of Physnet.
    Code of 'Remote Photoplethysmograph Signal Measurement from Facial Videos Using Spatio-Temporal Networks' 
    source: https://github.com/ZitongYu/PhysNet/blob/master/NegPearsonLoss.py
    """
    
    def __init__(self):
        super(Neg_Pearson, self).__init__()
        return


    def forward(self, preds, labels):       
        loss = 0
        for i in range(preds.shape[0]):
            sum_x = torch.sum(preds[i])               
            sum_y = torch.sum(labels[i])             
            sum_xy = torch.sum(preds[i]*labels[i])       
            sum_x2 = torch.sum(torch.pow(preds[i],2))  
            sum_y2 = torch.sum(torch.pow(labels[i],2)) 
            N = preds.shape[1]
            pearson = (N*sum_xy - sum_x*sum_y)/(torch.sqrt((N*sum_x2 - torch.pow(sum_x,2))*(N*sum_y2 - torch.pow(sum_y,2))))
            loss += 1 - pearson
            
            
        loss = loss/preds.shape[0]
        return loss






## Training utilities

Seed, train/val split, HR-MAE, best-checkpoint saver, early-stopping, CSV logger.


In [ ]:
# === Training utilities (shared across notebooks) ===
# seed, train/val split, HR-MAE, best-checkpoint saver, early-stopping, CSV logger.
import os
import random as _random
import csv as _csv
import time as _time
import numpy as _np
import torch as _torch
from scipy.signal import periodogram as _periodogram


def set_seed(seed: int = 42):
    """Set seeds for reproducibility across random, numpy, torch (CPU + CUDA)."""
    _random.seed(seed)
    _np.random.seed(seed)
    _torch.manual_seed(seed)
    _torch.cuda.manual_seed_all(seed)
    _torch.backends.cudnn.deterministic = True
    _torch.backends.cudnn.benchmark = False


def train_val_split(dataset, val_ratio: float = 0.2, seed: int = 42):
    """Random split returning (train_subset, val_subset) with a seeded generator."""
    n_total = len(dataset)
    n_val = max(1, int(n_total * val_ratio))
    n_train = n_total - n_val
    g = _torch.Generator().manual_seed(seed)
    return _torch.utils.data.random_split(dataset, [n_train, n_val], generator=g)


def compute_hr_fft(signal_1d, fps: int = 30, lo_hz: float = 0.6, hi_hz: float = 3.3) -> float:
    """Peak-frequency HR (bpm) of a 1-D signal via periodogram, restricted to a band."""
    sig = signal_1d.detach().cpu().numpy() if _torch.is_tensor(signal_1d) else _np.asarray(signal_1d)
    sig = sig.astype(_np.float64).ravel()
    if sig.size < 8 or sig.std() < 1e-8:
        return 0.0
    sig = sig - sig.mean()
    freqs, psd = _periodogram(sig, fs=fps)
    band = (freqs >= lo_hz) & (freqs <= hi_hz)
    if not band.any():
        return 0.0
    return float(freqs[band][psd[band].argmax()] * 60.0)


def compute_hr_mae_batch(preds, labels, fps: int = 30) -> float:
    """Mean absolute HR error (bpm) over a batch.  preds/labels: (N, T) or (T,)."""
    if preds.dim() == 1:
        preds = preds.unsqueeze(0)
    if labels.dim() == 1:
        labels = labels.unsqueeze(0)
    errs = []
    for i in range(preds.shape[0]):
        hr_p = compute_hr_fft(preds[i], fps)
        hr_l = compute_hr_fft(labels[i], fps)
        errs.append(abs(hr_p - hr_l))
    return float(_np.mean(errs)) if errs else 0.0


class BestCheckpointSaver:
    """Save the model's state_dict whenever a tracked metric improves."""
    def __init__(self, path: str, mode: str = "min"):
        assert mode in ("min", "max")
        self.path = path
        self.mode = mode
        self.best = float("inf") if mode == "min" else -float("inf")
        os.makedirs(os.path.dirname(os.path.abspath(path)) or ".", exist_ok=True)

    def step(self, model, metric: float) -> bool:
        improved = (metric < self.best) if self.mode == "min" else (metric > self.best)
        if improved and not (metric != metric):  # reject NaN
            self.best = metric
            _torch.save(model.state_dict(), self.path)
            return True
        return False


class EarlyStopping:
    """Stop training when the tracked metric stops improving for `patience` epochs."""
    def __init__(self, patience: int = 5, mode: str = "min", min_delta: float = 0.0):
        assert mode in ("min", "max")
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.best = float("inf") if mode == "min" else -float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, metric: float) -> bool:
        improved = (
            (self.mode == "min" and metric < self.best - self.min_delta)
            or (self.mode == "max" and metric > self.best + self.min_delta)
        )
        if improved:
            self.best = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


class MetricLogger:
    """Append per-epoch metrics to a CSV.  Creates the file with headers on init."""
    def __init__(self, csv_path: str, fieldnames=None):
        self.csv_path = csv_path
        self.fieldnames = list(fieldnames) if fieldnames else [
            "epoch", "train_loss", "val_loss", "val_hr_mae", "lr", "time_sec"
        ]
        os.makedirs(os.path.dirname(os.path.abspath(csv_path)) or ".", exist_ok=True)
        with open(csv_path, "w", newline="") as f:
            _csv.DictWriter(f, fieldnames=self.fieldnames).writeheader()

    def log(self, **row):
        with open(self.csv_path, "a", newline="") as f:
            _csv.DictWriter(f, fieldnames=self.fieldnames).writerow(
                {k: row.get(k, "") for k in self.fieldnames}
            )


# Seed the run for reproducibility
set_seed(42)
print("Training utils ready  |  seed=42  |  HR-MAE band: 0.6-3.3 Hz (36-198 bpm)")


In [ ]:
# ----- Dataset -----
class PhysNetDataset(Dataset):
    def __init__(self, input_files):
        self.inputs = sorted(input_files)
        self.labels = [
            f.replace("input", "label")
            for f in self.inputs
        ]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))   # (T, H, W, 3)
        label = np.float32(np.load(self.labels[index]))   # (T,)

        # NDHWC -> NCDHW: transpose (3, 0, 1, 2)
        data = np.transpose(data, (3, 0, 1, 2))  # (3, T, H, W)

        fname      = os.path.basename(self.inputs[index])
        split_idx  = fname.index("_")
        subject_id = fname[:split_idx]                     # e.g. "S000"
        chunk_id   = fname[split_idx + 6:].split(".")[0]  # +6 skips "_input"

        return data, label, subject_id, chunk_id

In [ ]:
# ----- DataLoader -----
all_input_files = glob.glob(os.path.join(PREPROCESSED_PATH, "*", "*_input*.npy"))
dataset = PhysNetDataset(all_input_files)
print(f"Total clips in dataset: {len(dataset)}")

# Split dataset into 80% train and 20% validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
print(f"Train batches: {len(train_loader)}, Validation batches: {len(val_loader)}")

In [ ]:
# Optimized training (HR-MAE on val + best-save + early-stop + grad-clip + CSV log)
SEED = 42
PATIENCE = 5
VIDEO_FPS = 30
LOG_PATH = os.path.join(REPO_ROOT, "results/Headmotion/groupC/train_logs/PhysNet.csv")

model = PhysNet_padding_Encoder_Decoder_MAX(frames=CHUNK_LENGTH).to(DEVICE)
loss_fn = Neg_Pearson()
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader)
)
saver   = BestCheckpointSaver(OUTPUT_WEIGHTS_PATH, mode="min")
stopper = EarlyStopping(patience=PATIENCE, mode="min")
logger  = MetricLogger(LOG_PATH)
print(f"Save best -> {OUTPUT_WEIGHTS_PATH}  |  Log -> {LOG_PATH}")


for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    train_loss_sum, n_train = 0.0, 0
    tbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", ncols=90)
    for batch in tbar:
        optimizer.zero_grad()
        data = batch[0].float().to(DEVICE, non_blocking=True)
        label = batch[1].float().to(DEVICE, non_blocking=True)
        rPPG, _, _, _ = model(data)
        rPPG  = (rPPG  - rPPG.mean())  / (rPPG.std()  + 1e-7)
        label = (label - label.mean()) / (label.std() + 1e-7)
        loss = loss_fn(rPPG, label)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        train_loss_sum += loss.item()
        n_train += 1
        tbar.set_postfix(loss=f"{loss.item():.4f}")
    train_loss = train_loss_sum / max(1, n_train)

    model.eval()
    val_loss_sum, val_hr_mae_sum, n_val = 0.0, 0.0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]  ", ncols=90):
            data = batch[0].float().to(DEVICE, non_blocking=True)
            label = batch[1].float().to(DEVICE, non_blocking=True)
            rPPG, _, _, _ = model(data)
            rPPG  = (rPPG  - rPPG.mean())  / (rPPG.std()  + 1e-7)
            label_n = (label - label.mean()) / (label.std() + 1e-7)
            val_loss_sum += loss_fn(rPPG, label_n).item()
            val_hr_mae_sum += compute_hr_mae_batch(rPPG, label, fps=VIDEO_FPS) * rPPG.shape[0]
            n_val += rPPG.shape[0]
    val_loss   = val_loss_sum / max(1, len(val_loader))
    val_hr_mae = val_hr_mae_sum / max(1, n_val)
    elapsed = time.time() - t0
    cur_lr = optimizer.param_groups[0]["lr"]

    improved = saver.step(model, val_hr_mae)
    marker = " <- best" if improved else ""
    print(f"Epoch {epoch+1:2d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"val_HR_MAE={val_hr_mae:.2f} bpm  lr={cur_lr:.2e}  ({elapsed:.1f}s){marker}")
    logger.log(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss,
               val_hr_mae=val_hr_mae, lr=cur_lr, time_sec=elapsed)

    if stopper.step(val_hr_mae):
        print(f"Early stopping at epoch {epoch+1}")
        break

print(f"\nBest val HR-MAE: {saver.best:.2f} bpm  ->  {OUTPUT_WEIGHTS_PATH}")
